# XGBoost — Nowcast độ mặn tuần tại trạm (ĐBSCL)
chạy trên `weekly_ml_dataset_v2_leftjoin.csv`.

Điểm khác bản cũ (`train_week.ipynb`):
1. **Thước đo = walk-forward theo thời gian** (train 2020–21 / val 2022 / test 2023).
2. **Dùng hết 1960 dòng mặn** (left-join phổ, phổ=NaN khi thiếu ảnh) thay vì 632 dòng inner-join.
3. **Hai model tách biệt**: hồi quy `sal_mean` (dòng `mean_reliable`) + phân loại ngưỡng 1/4 g/L (toàn bộ dòng, `sample_weight`).
4. Feature: thêm `lag1/lag2`, mùa vụ cyclic, `completeness_pct`, `source_format`.
5. Bỏ `year` (không ngoại suy được), cap outlier, target `mean` thay `max`.

In [1]:
#%pip install xgboost scikit-learn pandas numpy

In [2]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings("ignore")
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             precision_recall_fscore_support)

# ---- Cấu hình ----
CSV_PATH = "../../Data/xgboost/processed/weekly_ml_dataset_v2_leftjoin.csv"
SALINITY_CAP = 40.0     # cap giá trị phi vật lý
USE_COMPLETENESS = False # toggle ablation: tắt để kiểm tra completeness_pct có phải proxy protocol đo không
def rmse(a,b): return np.sqrt(mean_squared_error(a,b))

## 1. Load + làm sạch
Ép comma-decimal tự động, cap outlier độ mặn.

In [3]:
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
KEEP_STR = {"station_id","date","station_name_first","river_name_first","source_format","week_date"}
for c in df.columns:
    if c not in KEEP_STR and not pd.api.types.is_numeric_dtype(df[c]):
        cc = pd.to_numeric(df[c].astype(str).str.replace(",",".",regex=False), errors="coerce")
        if cc.notna().mean() > 0.8: df[c] = cc
for c in ["salinity_max_max","salinity_max_mean","salinity_max_min"]:
    df[c] = df[c].clip(upper=SALINITY_CAP)
df = df.sort_values(["station_id","date"]).reset_index(drop=True)
print(df.shape, "| có phổ:", int(df["s2_available"].sum()), "| phổ=NaN:", int((df["s2_available"]==0).sum()))

(1960, 97) | có phổ: 632 | phổ=NaN: 1328


## 2. Feature engineering (Bước 5)
- `lag1/lag2`: độ mặn tuần trước, hai tuần trước (proxy quán tính).
- Mùa vụ **cyclic** sin/cos (day-of-year + week-of-year) thay số nguyên.
- `source_format` one-hot (kèm `unknown` cho 375 dòng chưa verify được nguồn).
- `completeness_pct`: feature phụ để model tách ảnh hưởng cách đo.

In [4]:
g = df.groupby("station_id")
df["salinity_lag1"] = g["salinity_max_mean"].shift(1)
df["salinity_lag2"] = g["salinity_max_mean"].shift(2)
doy = df["day_of_year"]
df["doy_sin"]  = np.sin(2*np.pi*doy/365.25); df["doy_cos"]  = np.cos(2*np.pi*doy/365.25)
df["week_sin"] = np.sin(2*np.pi*df["week"]/52); df["week_cos"] = np.cos(2*np.pi*df["week"]/52)
df["source_format"] = df["source_format"].fillna("unknown")
sf = pd.get_dummies(df["source_format"], prefix="sf")
df = pd.concat([df, sf], axis=1)

STATIC   = ["lon_first","lat_first","DEM_first","Distance_to_River_first","Distance_to_Coast_first"]
SEASON   = ["doy_sin","doy_cos","week_sin","week_cos"]
WEATHER  = ["temperature_2m_c_mean","precipitation_mm_sum","rainy_day_sum","potential_evaporation_mm_sum",
            "runoff_mm_sum","runoff_mm_max","wind_speed_10m_ms_mean","wind_speed_10m_ms_max",
            "surface_pressure_hpa_min","soil_moisture_layer1_vol_mean","solar_radiation_mj_m2_sum"]
SPECTRAL = ["MNDWI_median_week","NDWI_median_week","NDVI_median_week","B11_median_week","B12_median_week",
            "Red_SWIR1_median_week","Red_SWIR2_median_week","BGRratio_median_week","NDCI_median_week"]
LAG      = ["salinity_lag1","salinity_lag2"]
QUALITY  = (["completeness_pct"] if USE_COMPLETENESS else []) + list(sf.columns)
FEATURES = [c for c in STATIC+SEASON+WEATHER+SPECTRAL+QUALITY if c in df.columns]
print(len(FEATURES), "features")
FEATURES.remove('sf_hourly_dense') if 'sf_hourly_dense' in FEATURES else None
FEATURES.remove('sf_hourly_sparse_2020') if 'sf_hourly_sparse_2020' in FEATURES else None
FEATURES.remove('sf_unknown') if 'sf_unknown' in FEATURES else None
print(len(FEATURES), "features")

32 features
29 features


In [5]:
print("Các cột feature:", FEATURES)

Các cột feature: ['lon_first', 'lat_first', 'DEM_first', 'Distance_to_River_first', 'Distance_to_Coast_first', 'doy_sin', 'doy_cos', 'week_sin', 'week_cos', 'temperature_2m_c_mean', 'precipitation_mm_sum', 'rainy_day_sum', 'potential_evaporation_mm_sum', 'runoff_mm_sum', 'runoff_mm_max', 'wind_speed_10m_ms_mean', 'wind_speed_10m_ms_max', 'surface_pressure_hpa_min', 'soil_moisture_layer1_vol_mean', 'solar_radiation_mj_m2_sum', 'MNDWI_median_week', 'NDWI_median_week', 'NDVI_median_week', 'B11_median_week', 'B12_median_week', 'Red_SWIR1_median_week', 'Red_SWIR2_median_week', 'BGRratio_median_week', 'NDCI_median_week']


## 3. Walk-forward split theo thời gian (Bước 7)
train 2020–2021 · val 2022 (early stopping) · test 2023 — đồng bộ với LSTM để so sánh công bằng.

In [6]:
tr  = df["year"].isin([2020,2021])
va  = df["year"]==2022
teY = df["year"]==2023
print(f"train={tr.sum()}  val={va.sum()}  test={teY.sum()}")

train=1010  val=467  test=483


## 4. MODEL 1 — Hồi quy `salinity_max_mean` (Bước 9)
Chỉ huấn luyện/đánh giá trên dòng `mean_reliable=True` (đủ ngày đo trong tuần). Target log1p vì lệch phải.

In [7]:
rel = df["mean_reliable"]==True
mtr, mva, mte = tr&rel, va&rel, teY&rel
yr = np.log1p(df["salinity_max_mean"])

regr = XGBRegressor(objective="reg:squarederror", learning_rate=0.03, max_depth=3,
                    n_estimators=3000, early_stopping_rounds=60, subsample=0.8,
                    colsample_bytree=0.8, random_state=42, min_child_weight=3,
                    reg_lambda=5.0, reg_alpha=0.5)
regr.fit(df.loc[mtr,FEATURES], yr[mtr],
         eval_set=[(df.loc[mva,FEATURES], yr[mva])], verbose=False)

pred = regr.predict(df.loc[mte,FEATURES]); yt = np.expm1(yr[mte]); yp = np.expm1(pred)
print(f"n_train={mtr.sum()} n_val={mva.sum()} n_test={mte.sum()}")
print(f"R2={r2_score(yr[mte],pred):.3f} | RMSE={rmse(yt,yp):.2f} g/L | MAE={mean_absolute_error(yt,yp):.2f} g/L")

pred_train = regr.predict(df.loc[mtr, FEATURES])

# ===== Validation =====
pred_val = regr.predict(df.loc[mva, FEATURES])

# ===== Test =====
pred_test = regr.predict(df.loc[mte, FEATURES])

# R² trên log scale
train_r2 = r2_score(yr[mtr], pred_train)
val_r2   = r2_score(yr[mva], pred_val)
test_r2  = r2_score(yr[mte], pred_test)

print(f"Train R² = {train_r2:.3f}")
print(f"Val   R² = {val_r2:.3f}")
print(f"Test  R² = {test_r2:.3f}")
print(regr.best_iteration)
print(regr.best_score)
# Chuyển về g/L
y_train = np.expm1(yr[mtr])
y_val   = np.expm1(yr[mva])
y_test  = np.expm1(yr[mte])

pred_train_g = np.expm1(pred_train)
pred_val_g   = np.expm1(pred_val)
pred_test_g  = np.expm1(pred_test)

print("Train RMSE:", np.sqrt(mean_squared_error(y_train, pred_train_g)))
print("Val   RMSE:", np.sqrt(mean_squared_error(y_val, pred_val_g)))
print("Test  RMSE:", np.sqrt(mean_squared_error(y_test, pred_test_g)))

print("Train MAE :", mean_absolute_error(y_train, pred_train_g))
print("Val   MAE :", mean_absolute_error(y_val, pred_val_g))
print("Test  MAE :", mean_absolute_error(y_test, pred_test_g))

n_train=571 n_val=294 n_test=252
R2=0.761 | RMSE=2.11 g/L | MAE=1.62 g/L
Train R² = 0.928
Val   R² = 0.564
Test  R² = 0.761
364
0.5565398880235685
Train RMSE: 2.4948472610657118
Val   RMSE: 2.3974303903249736
Test  RMSE: 2.114765399622223
Train MAE : 1.3407542508779047
Val   MAE : 1.8601609767023641
Test  MAE : 1.6210404776131113


### 4b. Sai số theo `source_format` (Bước 10 — kiểm tra bias theo loại trạm)

In [8]:
for s in sorted(df.loc[mte,"source_format"].unique()):
    idx = mte & (df["source_format"]==s)
    if idx.sum()>=5:
        e = mean_absolute_error(np.expm1(yr[idx]), np.expm1(regr.predict(df.loc[idx,FEATURES])))
        print(f"  {s:<20} n={idx.sum():3d}  MAE={e:.2f} g/L")

  hourly_dense         n=252  MAE=1.62 g/L


### 4c. Feature importance
> **Lưu ý:** nếu `completeness_pct` đứng top, chạy lại với `USE_COMPLETENESS=False` (cell 1) để kiểm tra
> nó là tín hiệu thật hay chỉ là *proxy của protocol đo* (đo dày hơn vào mùa mặn).

In [9]:
imp = (pd.DataFrame({"feature":FEATURES, "importance":regr.feature_importances_})
       .sort_values("importance", ascending=False).head(20).reset_index(drop=True))
imp

,feature,importance
0,rainy_day_sum,0.152580
1,Distance_to_Coast_first,0.150200
2,wind_speed_10m_ms_mean,0.100948
3,lat_first,0.075790
4,wind_speed_10m_ms_max,0.065933
5,lon_first,0.061078
6,runoff_mm_sum,0.046885
7,DEM_first,0.040540
8,solar_radiation_mj_m2_sum,0.039750
9,Distance_to_River_first,0.034598


## 4d. Baseline so sánh (Bước 10b)
So sánh Model 1 (XGBoost hồi quy) với 2 baseline, cùng chạy trên `mtr/mva/mte` và cùng `FEATURES` (không lag):
1. **Station-week climatology** — trung bình `salinity_max_mean` theo (station_id, week) tính trên train; fallback về trung bình theo week toàn cục, rồi về trung bình chung nếu vẫn thiếu.
2. **Ridge regression** trên đúng bộ `FEATURES` hiện tại (cùng target `log1p`) — để tách phần đóng góp của việc học phi tuyến/tương tác (XGBoost) so với một model tuyến tính đơn giản trên cùng input.

Lưu ý: cả hai baseline đều **không dùng `salinity_lag1/lag2`**, khớp với mục tiêu Sentinel-only (áp dụng được cả ở trạm chưa có lịch sử quan trắc).

In [10]:
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer

# ============ Baseline A: Station-week climatology ============
# Trung bình salinity_max_mean theo (station_id, week) tính trên train (mtr).
# Fallback: nếu (station, week) chưa thấy trong train -> trung bình theo week toàn cục -> trung bình chung.
clim_station = df.loc[mtr].groupby(["station_id", "week"])["salinity_max_mean"].mean()
clim_global  = df.loc[mtr].groupby("week")["salinity_max_mean"].mean()
overall_mean = df.loc[mtr, "salinity_max_mean"].mean()

def climatology_predict(idx):
    preds = []
    for sid, wk in zip(df.loc[idx, "station_id"], df.loc[idx, "week"]):
        if (sid, wk) in clim_station.index:
            preds.append(clim_station.loc[(sid, wk)])
        elif wk in clim_global.index:
            preds.append(clim_global.loc[wk])
        else:
            preds.append(overall_mean)
    return np.array(preds)

pred_clim_test = climatology_predict(mte)

# ============ Baseline B: Ridge regression trên FEATURES hiện tại ============
# Cùng target log1p, cùng split mtr (train) / mte (test) với XGBoost để so sánh công bằng.
# Ridge (khác XGBoost) không tự xử lý NaN -> impute median tính trên train (mtr) rồi áp cho cả train/test,
# tránh rò rỉ thông tin từ val/test vào bước impute.
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(df.loc[mtr, FEATURES])
X_test_imp  = imputer.transform(df.loc[mte, FEATURES])

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_imp, yr[mtr])
pred_ridge_test = np.expm1(ridge.predict(X_test_imp))

# ============ Bảng so sánh 3 dòng (đều đánh giá trên test 2023, thang g/L) ============
def eval_row(name, y_true, y_pred):
    return {
        "model": name,
        "RMSE (g/L)": rmse(y_true, y_pred),
        "MAE (g/L)": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

baseline_results = pd.DataFrame([
    eval_row("Station-week climatology", y_test, pred_clim_test),
    eval_row("Ridge regression (cùng FEATURES, không lag)", y_test, pred_ridge_test),
    eval_row("XGBoost (Model 1 hiện tại)", y_test, pred_test_g),
])
baseline_results

,model,RMSE (g/L),MAE (g/L),R2
0,Station-week climatology,3.265437,2.537691,0.503294
1,"Ridge regression (cùng FEATURES, không lag)",3.276642,2.501860,0.499880
2,XGBoost (Model 1 hiện tại),2.114765,1.621040,0.791675


## 4e. Kiểm tra khả năng ngoại suy không gian — trạm/toạ độ chưa từng thấy (Bước 10c)
Split hiện tại (`tr/va/teY`) chỉ chia theo **năm**, chưa kiểm chứng model có dùng được cho toạ độ hoàn toàn mới hay không (mục tiêu ban đầu: áp dụng Sentinel cho toàn vùng, kể cả nơi chưa có trạm quan trắc). Gồm 2 bước:
1. **Kiểm tra overlap trạm** giữa train/val/test — nếu trùng gần như hoàn toàn, R² hiện tại không nói lên gì về khả năng ngoại suy không gian.
2. **Leave-station-out CV**: giữ hẳn một nhóm trạm ra khỏi train/val (chưa từng xuất hiện), huấn luyện lại, rồi so sánh hiệu năng trên tập test giữa nhóm trạm quen (control) và nhóm trạm hoàn toàn lạ.

In [11]:
# ============ Kiểm tra overlap trạm giữa train / val / test ============
stations_train = set(df.loc[tr, "station_id"])
stations_val   = set(df.loc[va, "station_id"])
stations_test  = set(df.loc[teY, "station_id"])

print(f"Số trạm trong train (2020-21): {len(stations_train)}")
print(f"Số trạm trong val   (2022)   : {len(stations_val)}")
print(f"Số trạm trong test  (2023)   : {len(stations_test)}")
print()
unseen_in_test = stations_test - (stations_train | stations_val)
print(f"Số trạm trong test hoàn toàn KHÔNG xuất hiện ở train+val: {len(unseen_in_test)} / {len(stations_test)}")
if len(unseen_in_test) == 0:
    print("-> R² hiện tại của Model 1 KHÔNG kiểm chứng khả năng ngoại suy sang toạ độ mới:")
    print("   mọi trạm test đều đã 'quen mặt' với model từ giai đoạn train/val.")


Số trạm trong train (2020-21): 30
Số trạm trong val   (2022)   : 21
Số trạm trong test  (2023)   : 21

Số trạm trong test hoàn toàn KHÔNG xuất hiện ở train+val: 0 / 21
-> R² hiện tại của Model 1 KHÔNG kiểm chứng khả năng ngoại suy sang toạ độ mới:
   mọi trạm test đều đã 'quen mặt' với model từ giai đoạn train/val.


In [12]:
# ============ Leave-station-out CV: mô phỏng toạ độ chưa từng thấy ============
# Giữ hẳn 1 nhóm trạm ra khỏi TOÀN BỘ train/val (không chỉ test), huấn luyện lại từ đầu,
# rồi so sánh hiệu năng trên test giữa: trạm ĐÃ thấy (control) vs trạm CHƯA từng thấy (hold-out).
RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

all_stations = np.array(sorted(df.loc[rel, "station_id"].unique()))
rng.shuffle(all_stations)

N_HOLDOUT = max(1, int(round(0.25 * len(all_stations))))
held_out_stations = set(all_stations[:N_HOLDOUT])

print(f"Tổng số trạm (mean_reliable): {len(all_stations)}")
print(f"Số trạm giữ lại làm 'chưa từng thấy' (hold-out): {N_HOLDOUT}")
print(f"Số trạm dùng để train/val: {len(all_stations) - N_HOLDOUT}")

is_holdout = df["station_id"].isin(held_out_stations)

# Train/val: chỉ dùng trạm KHÔNG bị hold-out, vẫn giữ nguyên walk-forward theo năm
lso_train = mtr & ~is_holdout
lso_val   = mva & ~is_holdout

# Test: tách 2 nhóm trong cùng năm 2023
lso_test_seen   = mte & ~is_holdout   # trạm quen -> control
lso_test_unseen = mte &  is_holdout   # trạm hoàn toàn chưa thấy trong train/val

print(f"n_train={lso_train.sum()}  n_val={lso_val.sum()}  "
      f"n_test_seen={lso_test_seen.sum()}  n_test_unseen={lso_test_unseen.sum()}")

regr_lso = XGBRegressor(objective="reg:squarederror", learning_rate=0.03, max_depth=3,
                        n_estimators=3000, early_stopping_rounds=60, subsample=0.8,
                        colsample_bytree=0.8, random_state=42, min_child_weight=3,
                        reg_lambda=5.0, reg_alpha=0.5)
regr_lso.fit(df.loc[lso_train, FEATURES], yr[lso_train],
             eval_set=[(df.loc[lso_val, FEATURES], yr[lso_val])], verbose=False)

def eval_group(name, idx):
    if idx.sum() == 0:
        print(f"  {name}: không có dòng nào, bỏ qua")
        return None
    y_true = np.expm1(yr[idx])
    y_pred = np.expm1(regr_lso.predict(df.loc[idx, FEATURES]))
    return {
        "nhóm": name,
        "n": int(idx.sum()),
        "RMSE (g/L)": rmse(y_true, y_pred),
        "MAE (g/L)": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

lso_results = pd.DataFrame([r for r in [
    eval_group("Test - trạm ĐÃ thấy trong train (control)", lso_test_seen),
    eval_group("Test - trạm CHƯA từng thấy (hold-out)", lso_test_unseen),
] if r is not None])
lso_results


Tổng số trạm (mean_reliable): 30
Số trạm giữ lại làm 'chưa từng thấy' (hold-out): 8
Số trạm dùng để train/val: 22
n_train=415  n_val=210  n_test_seen=168  n_test_unseen=84


,nhóm,n,RMSE (g/L),MAE (g/L),R2
0,Test - trạm ĐÃ thấy trong train (control),168,2.382612,1.822200,0.774585
1,Test - trạm CHƯA từng thấy (hold-out),84,5.342283,4.344006,-1.181874


> **Lưu ý:** kết quả trên chỉ ứng với 1 cách chia trạm ngẫu nhiên (seed=42). Nếu số trạm không quá nhiều, nên chạy lại với vài `RANDOM_STATE` khác nhau (hoặc làm K-fold theo trạm) rồi lấy trung bình, để tránh kết luận vội vàng chỉ dựa trên 1 lần chia may/rủi.

## 5. MODEL 2 — Phân loại vượt ngưỡng 1 & 4 g/L (Bước 9)
Dùng **`salinity_max_max`** (đỉnh tuần) và **toàn bộ dòng** kể cả phổ=NaN và trạm thưa —
với `sample_weight` thấp cho dòng chất lượng kém thay vì loại bỏ. Đây là đầu ra vận hành cho cảnh báo ranh mặn.

In [13]:
w = np.where(df["mean_reliable"]==True, 1.0,
     np.where(df["mean_reliable"]==False, 0.4, 0.5))   # NaN (chưa verify) -> 0.5

for thr in [1.0, 4.0]:
    yb = (df["salinity_max_max"] >= thr).astype(int)
    clf = XGBClassifier(objective="binary:logistic", learning_rate=0.05, max_depth=4,
                        n_estimators=800, subsample=0.8, colsample_bytree=0.8,
                        eval_metric="logloss", random_state=42)
    clf.fit(df.loc[tr|va,FEATURES], yb[tr|va], sample_weight=w[tr|va])
    pb = clf.predict(df.loc[teY,FEATURES])
    p,r,f,_ = precision_recall_fscore_support(yb[teY], pb, average="binary", zero_division=0)
    print(f"≥{thr:g} g/L  (test n={teY.sum()}, dương={int(yb[teY].sum())}):  P={p:.2f}  R={r:.2f}  F1={f:.2f}")

≥1 g/L  (test n=483, dương=384):  P=0.91  R=0.98  F1=0.94
≥4 g/L  (test n=483, dương=277):  P=0.82  R=0.92  F1=0.87


## 6. Export kết quả → `models/xgboost/`
Lưu metrics (hồi quy + phân loại), feature importance, bảng so sánh baseline/leave-station-out, và model (`.json`) để commit vào `models/xgboost/`

In [14]:
# ============================================================
# EXPORT — lưu metrics / feature importance / model ra models/xgboost/
# ============================================================
import os, json
OUT_DIR = "../../models/xgboost"
os.makedirs(OUT_DIR, exist_ok=True)

# 1) Metrics hồi quy (Model 1): train / val / test
reg_metrics = pd.DataFrame([
    {"split":"train","n":int(mtr.sum()),"R2_log":r2_score(yr[mtr],pred_train),
     "RMSE_gL":rmse(y_train,pred_train_g),"MAE_gL":mean_absolute_error(y_train,pred_train_g)},
    {"split":"val","n":int(mva.sum()),"R2_log":r2_score(yr[mva],pred_val),
     "RMSE_gL":rmse(y_val,pred_val_g),"MAE_gL":mean_absolute_error(y_val,pred_val_g)},
    {"split":"test","n":int(mte.sum()),"R2_log":r2_score(yr[mte],pred_test),
     "RMSE_gL":rmse(y_test,pred_test_g),"MAE_gL":mean_absolute_error(y_test,pred_test_g)},
])
reg_metrics.to_csv(f"{OUT_DIR}/metrics_regression.csv", index=False)

# 2) Feature importance
imp.to_csv(f"{OUT_DIR}/feature_importance.csv", index=False)

# 3) Bảng so sánh baseline + leave-station-out (nếu đã chạy các cell trên)
try: baseline_results.to_csv(f"{OUT_DIR}/baseline_comparison.csv", index=False)
except NameError: pass
try: lso_results.to_csv(f"{OUT_DIR}/leave_station_out.csv", index=False)
except NameError: pass

# 4) Phân loại (Model 2): fit lại 2 ngưỡng -> lưu metrics + model
w = np.where(df["mean_reliable"]==True, 1.0,
     np.where(df["mean_reliable"]==False, 0.4, 0.5))
clf_rows = []
for thr in [1.0, 4.0]:
    yb = (df["salinity_max_max"] >= thr).astype(int)
    clf = XGBClassifier(objective="binary:logistic", learning_rate=0.05, max_depth=4,
                        n_estimators=800, subsample=0.8, colsample_bytree=0.8,
                        eval_metric="logloss", random_state=42)
    clf.fit(df.loc[tr|va,FEATURES], yb[tr|va], sample_weight=w[tr|va])
    pb = clf.predict(df.loc[teY,FEATURES])
    p,r,f,_ = precision_recall_fscore_support(yb[teY], pb, average="binary", zero_division=0)
    clf_rows.append({"threshold_gL":thr,"n_test":int(teY.sum()),"n_positive":int(yb[teY].sum()),
                     "precision":round(p,4),"recall":round(r,4),"f1":round(f,4)})
    clf.save_model(f"{OUT_DIR}/xgb_classifier_{int(thr)}gL.json")
pd.DataFrame(clf_rows).to_csv(f"{OUT_DIR}/metrics_classification.csv", index=False)

# 5) Model hồi quy + danh sách feature
regr.save_model(f"{OUT_DIR}/xgb_regression.json")
json.dump(FEATURES, open(f"{OUT_DIR}/feature_list.json","w"), ensure_ascii=False, indent=2)

print("Đã lưu vào", OUT_DIR, ":")
for fn in sorted(os.listdir(OUT_DIR)): print("   -", fn)

Đã lưu vào ../../models/xgboost :
   - baseline_comparison.csv
   - feature_importance.csv
   - feature_list.json
   - leave_station_out.csv
   - metrics_classification.csv
   - metrics_regression.csv
   - xgb_classifier_1gL.json
   - xgb_classifier_4gL.json
   - xgb_regression.json
